# 5C0F

In [1]:
import sys
sys.path.append("../..")

from data.training_simulations import save_training_xtcs

conditions = [
    "shuffle_allframes",
    "shuffle_not_allframes",
    "not_shuffle_allframes",
    "not_shuffle_not_allframes"
]

for condition in conditions:
    save_training_xtcs(
        protein_id="5C0F",
        condition=condition,
        replicas=(0, 1, 2),
        fractions=(10, 20, 30, 40, 50, 60, 70),
        base_data_dir="../../data/palantir_data",
        output_dir="Training_XTCs",
        seed=42
    )


=== 5C0F | rep 0 | shuffle_allframes ===
Original frames: 1001
[Fraction 10] Seed: 52 | Train frames: 100 | Saved: Training_XTCs/shuffle_allframes/5C0F/5C0F_rep_0/fraction_10/training_5C0F_rep_0_frac_10.xtc
[Fraction 20] Seed: 62 | Train frames: 200 | Saved: Training_XTCs/shuffle_allframes/5C0F/5C0F_rep_0/fraction_20/training_5C0F_rep_0_frac_20.xtc
[Fraction 30] Seed: 72 | Train frames: 300 | Saved: Training_XTCs/shuffle_allframes/5C0F/5C0F_rep_0/fraction_30/training_5C0F_rep_0_frac_30.xtc
[Fraction 40] Seed: 82 | Train frames: 400 | Saved: Training_XTCs/shuffle_allframes/5C0F/5C0F_rep_0/fraction_40/training_5C0F_rep_0_frac_40.xtc
[Fraction 50] Seed: 92 | Train frames: 500 | Saved: Training_XTCs/shuffle_allframes/5C0F/5C0F_rep_0/fraction_50/training_5C0F_rep_0_frac_50.xtc
[Fraction 60] Seed: 102 | Train frames: 600 | Saved: Training_XTCs/shuffle_allframes/5C0F/5C0F_rep_0/fraction_60/training_5C0F_rep_0_frac_60.xtc
[Fraction 70] Seed: 112 | Train frames: 700 | Saved: Training_XTCs/shuf

In [3]:
import sys
sys.path.append("../..")

from run_pipeline import run_pipeline

protein_id = "5C0F"
base_data_dir = "../../data/palantir_data/5C0F"

pdb_file = f"{base_data_dir}/{protein_id}.pdb"
atom_selection = "backbone"

latent_dim = 16
epochs = 300
batch_size = 64
kl_beta = 5e-5
base_seed = 42
device = "cuda"

replicas = [0, 1, 2]

# Final selected model: M1_kl_annealing_sym
model_kwargs = {
    "encoder_dims": (512, 256, 128),
    "decoder_dims": (128, 256, 512),
    "use_residual": False,
    "use_attention": False,
    "use_layernorm": True,
    "activation": "gelu",
}

train_kwargs = {
    "lr": 5e-4,
    "use_kl_annealing": True,
}

experiments = [
    {
        "name": "shuffle_allframes",
        "use_all_frames": True,
        "shuffle_train": True,
        "base_out_dir": "../../Model_results_M1/shuffle_allframes/5C0F"
    },
    {
        "name": "shuffle_not_allframes",
        "use_all_frames": False,
        "shuffle_train": True,
        "base_out_dir": "../../Model_results_M1/shuffle_not_allframes/5C0F"
    },
    {
        "name": "not_shuffle_allframes",
        "use_all_frames": True,
        "shuffle_train": False,
        "base_out_dir": "../../Model_results_M1/not_shuffle_allframes/5C0F"
    },
    {
        "name": "not_shuffle_not_allframes",
        "use_all_frames": False,
        "shuffle_train": False,
        "base_out_dir": "../../Model_results_M1/not_shuffle_not_allframes/5C0F"
    }
]

condition_offsets = {
    "shuffle_allframes": 0,
    "shuffle_not_allframes": 1000,
    "not_shuffle_allframes": 2000,
    "not_shuffle_not_allframes": 3000,
}

for exp in experiments:

    print("\n**************************************")
    print(f"Condition: {exp['name']}")
    print(f"use_all_frames={exp['use_all_frames']} | shuffle_train={exp['shuffle_train']}")
    print("Model: M1_kl_annealing_sym")
    print("**************************************")

    for rep in replicas:
        xtc_file = f"{base_data_dir}/{protein_id}_rep_{rep}.xtc"
        out_dir = f"{exp['base_out_dir']}/{protein_id}_rep_{rep}"

        run_seed = base_seed + condition_offsets[exp["name"]] + rep * 100

        print(f"\n--- RUNNING {protein_id} replica {rep} ---")
        print(f"Seed: {run_seed}")
        print(f"Output: {out_dir}")

        run_pipeline(
            xtc_file=xtc_file,
            pdb_file=pdb_file,
            protein_id=protein_id,
            rep_num=rep,
            output_root=base_data_dir, 
            atom_selection=atom_selection,
            out_dir=out_dir,
            already_preprocessed=False,

            latent_dim=latent_dim,
            epochs=epochs,
            batch_size=batch_size,
            kl_beta=kl_beta,

            use_all_frames=exp["use_all_frames"],
            shuffle_train=exp["shuffle_train"],
            seed=run_seed,
            device=device,

            fractions=(10, 20, 30, 40, 50, 60, 70),
            temperature=1.5,
            target_n_frames=1000,
            generation_batch_size=64,
            max_generation_attempts=500000,
            rama_outlier_threshold=0.10,
            bond_bad_threshold=0.15,

            ablation_name="M1_kl_annealing_sym",
            model_kwargs=model_kwargs,
            train_kwargs=train_kwargs,
        )


**************************************
Condition: shuffle_allframes
use_all_frames=True | shuffle_train=True
Model: M1_kl_annealing_sym
**************************************

--- RUNNING 5C0F replica 0 ---
Seed: 42
Output: ../../Model_results_M1/shuffle_allframes/5C0F/5C0F_rep_0
[INFO] Frames: 1001, Atoms: 1548
[Fraction 10] Seed: 52 | Train: 100 | Val: 100 | Test: 200
[Fraction 20] Seed: 62 | Train: 200 | Val: 100 | Test: 200
[Fraction 30] Seed: 72 | Train: 300 | Val: 100 | Test: 200
[Fraction 40] Seed: 82 | Train: 400 | Val: 100 | Test: 200
[Fraction 50] Seed: 92 | Train: 500 | Val: 100 | Test: 200
[Fraction 60] Seed: 102 | Train: 600 | Val: 100 | Test: 200
[Fraction 70] Seed: 112 | Train: 700 | Val: 100 | Test: 200

--- RUNNING 5C0F replica 1 ---
Seed: 142
Output: ../../Model_results_M1/shuffle_allframes/5C0F/5C0F_rep_1
[INFO] Frames: 1001, Atoms: 1548
[Fraction 10] Seed: 152 | Train: 100 | Val: 100 | Test: 200
[Fraction 20] Seed: 162 | Train: 200 | Val: 100 | Test: 200
[Fraction 

In [4]:
import sys
sys.path.append("../..")
from model_analysis.outputs import *

base_dir = "../../Model_results_M1"

conditions = [
    "shuffle_allframes",
    "shuffle_not_allframes",
    "not_shuffle_allframes",
    "not_shuffle_not_allframes"
]

protein_id = "5C0F"

for condition in conditions:

    experiment_dirs = {
        "rep_0": f"{base_dir}/{condition}/{protein_id}/{protein_id}_rep_0",
        "rep_1": f"{base_dir}/{condition}/{protein_id}/{protein_id}_rep_1",
        "rep_2": f"{base_dir}/{condition}/{protein_id}/{protein_id}_rep_2",
    }

    # 1) Training dynamics (replica-wise)
    model_accuracy_results_replica_wise(experiment_dirs)

    # 2) Summary performance (mean ± std)
    model_performance_results_combined(experiment_dirs)

# Equal frame number

In [5]:
import sys
sys.path.append("../..")

from run_pipeline import run_pipeline

protein_id = "5C0F"
base_data_dir = "../../data/palantir_data_equal_frames/5C0F"

pdb_file = f"{base_data_dir}/backbone.pdb"
atom_selection = "backbone"

latent_dim = 16
epochs = 300
batch_size = 64
kl_beta = 5e-5
base_seed = 42
device = "cuda"

replicas = [0, 1, 2]

# Final selected model: M1_kl_annealing_sym
model_kwargs = {
    "encoder_dims": (512, 256, 128),
    "decoder_dims": (128, 256, 512),
    "use_residual": False,
    "use_attention": False,
    "use_layernorm": True,
    "activation": "gelu",
}

train_kwargs = {
    "lr": 5e-4,
    "use_kl_annealing": True,
}

experiments = [
    {
        "name": "shuffle_allframes",
        "use_all_frames": True,
        "shuffle_train": True,
        "base_out_dir": "../../Model_results_M1/Equal_frames/shuffle_allframes/5C0F"
    },
    {
        "name": "shuffle_not_allframes",
        "use_all_frames": False,
        "shuffle_train": True,
        "base_out_dir": "../../Model_results_M1/Equal_frames/shuffle_not_allframes/5C0F"
    },
    {
        "name": "not_shuffle_allframes",
        "use_all_frames": True,
        "shuffle_train": False,
        "base_out_dir": "../../Model_results_M1/Equal_frames/not_shuffle_allframes/5C0F"
    },
    {
        "name": "not_shuffle_not_allframes",
        "use_all_frames": False,
        "shuffle_train": False,
        "base_out_dir": "../../Model_results_M1/Equal_frames/not_shuffle_not_allframes/5C0F"
    }
]

condition_offsets = {
    "shuffle_allframes": 0,
    "shuffle_not_allframes": 1000,
    "not_shuffle_allframes": 2000,
    "not_shuffle_not_allframes": 3000,
}

for exp in experiments:

    print("\n**************************************")
    print(f"Condition: {exp['name']}")
    print(f"use_all_frames={exp['use_all_frames']} | shuffle_train={exp['shuffle_train']}")
    print("Model: M1_kl_annealing_sym")
    print("Equal-frame control experiment")
    print("**************************************")

    for rep in replicas:
        xtc_file = f"{base_data_dir}/{protein_id}_rep_{rep}_backbone.xtc"
        out_dir = f"{exp['base_out_dir']}/{protein_id}_rep_{rep}"

        run_seed = base_seed + condition_offsets[exp["name"]] + rep * 100

        print(f"\n--- RUNNING {protein_id} replica {rep} ---")
        print(f"Seed: {run_seed}")
        print(f"XTC: {xtc_file}")
        print(f"Output: {out_dir}")

        run_pipeline(
            xtc_file=xtc_file,
            pdb_file=pdb_file,
            protein_id=protein_id,
            rep_num=rep,
            output_root=base_data_dir, 
            atom_selection=atom_selection,
            out_dir=out_dir,
            already_preprocessed=True,

            latent_dim=latent_dim,
            epochs=epochs,
            batch_size=batch_size,
            kl_beta=kl_beta,

            use_all_frames=exp["use_all_frames"],
            shuffle_train=exp["shuffle_train"],
            seed=run_seed,
            device=device,

            fractions=(10, 20, 30, 40, 50, 60, 70),
            temperature=1.5,
            target_n_frames=1000,
            generation_batch_size=64,
            max_generation_attempts=500000,
            rama_outlier_threshold=0.10,
            bond_bad_threshold=0.15,

            ablation_name="M1_kl_annealing_sym",
            model_kwargs=model_kwargs,
            train_kwargs=train_kwargs,
        )


**************************************
Condition: shuffle_allframes
use_all_frames=True | shuffle_train=True
Model: M1_kl_annealing_sym
Equal-frame control experiment
**************************************

--- RUNNING 5C0F replica 0 ---
Seed: 42
XTC: ../../data/palantir_data_equal_frames/5C0F/5C0F_rep_0_backbone.xtc
Output: ../../Model_results_M1/Equal_frames/shuffle_allframes/5C0F/5C0F_rep_0
[INFO] Loaded preprocessed trajectory
[INFO] Frames: 875, Atoms: 1548
[Fraction 10] Seed: 52 | Train: 87 | Val: 87 | Test: 175
[Fraction 20] Seed: 62 | Train: 175 | Val: 87 | Test: 175
[Fraction 30] Seed: 72 | Train: 262 | Val: 87 | Test: 175
[Fraction 40] Seed: 82 | Train: 350 | Val: 87 | Test: 175
[Fraction 50] Seed: 92 | Train: 437 | Val: 87 | Test: 175
[Fraction 60] Seed: 102 | Train: 525 | Val: 87 | Test: 175
[Fraction 70] Seed: 112 | Train: 612 | Val: 87 | Test: 175

--- RUNNING 5C0F replica 1 ---
Seed: 142
XTC: ../../data/palantir_data_equal_frames/5C0F/5C0F_rep_1_backbone.xtc
Output: ../

In [6]:
from model_analysis.outputs import *

base_dir = "../../Model_results_M1/Equal_frames"

conditions = [
    "shuffle_allframes",
    "shuffle_not_allframes",
    "not_shuffle_allframes",
    "not_shuffle_not_allframes"
]

protein_id = "5C0F"

for condition in conditions:

    experiment_dirs = {
        "rep_0": f"{base_dir}/{condition}/{protein_id}/{protein_id}_rep_0",
        "rep_1": f"{base_dir}/{condition}/{protein_id}/{protein_id}_rep_1",
        "rep_2": f"{base_dir}/{condition}/{protein_id}/{protein_id}_rep_2",
    }

    # 1) Training dynamics (replica-wise)
    model_accuracy_results_replica_wise(experiment_dirs)

    # 2) Summary performance (mean ± std)
    model_performance_results_combined(experiment_dirs)